# 05. Long Meeting Transcription

## Goal

Split longer meeting audio into smaller chunks, transcribe each chunk, and keep timestamps with the transcript.

## Why This Step Matters

Real meeting audio is longer than one LOTUSDIS utterance. Splitting audio keeps ASR memory use predictable and makes the transcript easier to inspect by time range.

## Inputs

- Streamed `nectec/LOTUSDIS` validation samples from Hugging Face.
- `openai/whisper-small` baseline ASR model.
- Fixed-size audio chunking utilities from `src/transcript/`.

## Outputs

- A timestamped transcript table.
- Optional local JSON and Markdown transcript files under ignored `outputs/`.

## Setup

This debug notebook builds a pseudo long meeting by concatenating a few LOTUSDIS validation samples. Use a real audio file later after this flow is stable.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.transcript.segment import split_audio
from src.transcript.timestamps import format_time_range

MODEL_NAME = "openai/whisper-small"
MAX_SOURCE_SAMPLES = 3
CHUNK_SECONDS = 12.0
OVERLAP_SECONDS = 0.0
SAVE_OUTPUTS = False
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "long_meeting_debug"
DEVICE = 0 if torch.cuda.is_available() else -1
MODEL_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("Project root:", PROJECT_ROOT)
print("Device:", "cuda" if DEVICE == 0 else "cpu")

/home/lataeq/Code/Saranote/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /home/lataeq/Code/Saranote
Device: cuda


## Load Data

Load a few short samples and concatenate them into one audio array for a long-audio dry run.

In [2]:
def decode_audio(audio):
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        return samples.data.squeeze().detach().cpu().numpy(), samples.sample_rate

    return audio["array"], audio["sampling_rate"]


def collect_samples(split: str, count: int) -> list[dict[str, object]]:
    rows = []
    dataset = load_dataset("nectec/LOTUSDIS", split=split, streaming=True)

    for sample in dataset:
        audio_array, sampling_rate = decode_audio(sample["audio"])
        rows.append(
            {
                "audio": audio_array,
                "sampling_rate": sampling_rate,
                "sentence": sample["sentence"],
            }
        )

        if len(rows) == count:
            break

    return rows


source_samples = collect_samples("validation", MAX_SOURCE_SAMPLES)
sampling_rate = source_samples[0]["sampling_rate"]

if any(sample["sampling_rate"] != sampling_rate for sample in source_samples):
    raise ValueError("all debug samples must use the same sampling rate")

meeting_audio = np.concatenate([sample["audio"] for sample in source_samples])
reference_text = " ".join(sample["sentence"] for sample in source_samples)

print(f"Loaded {len(source_samples)} samples")
print(f"Duration: {len(meeting_audio) / sampling_rate:.2f}s")
print("Reference preview:", reference_text)

Loaded 3 samples
Duration: 6.83s
Reference preview: ครับสวัสดีครับผม M สามสิบแปดครับ สวัสดีครับ M สามสิบเก้าครับ สวัสดีค่ะ F สี่สิบสองค่ะ


## Run Experiment

Split the audio, run ASR on each chunk, and attach a timestamp range to every transcript row.

In [3]:
segments = split_audio(
    meeting_audio,
    sample_rate=sampling_rate,
    chunk_seconds=CHUNK_SECONDS,
    overlap_seconds=OVERLAP_SECONDS,
)

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    device=DEVICE,
    dtype=MODEL_DTYPE,
)

rows = []
for segment in segments:
    result = asr_pipe(
        {"array": segment.audio, "sampling_rate": sampling_rate},
        generate_kwargs={"language": "thai", "task": "transcribe"},
    )
    rows.append(
        {
            "chunk": segment.index,
            "timestamp": format_time_range(segment.start_seconds, segment.end_seconds),
            "text": result["text"].strip(),
        }
    )

transcript_df = pd.DataFrame(rows)
transcript_df

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1469.52it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.Suppres

,chunk,timestamp,text
0,1,00:00:00 - 00:00:06.830,สวัสดีครับ ผมม38ครับสวัสดีครับ ม39ครับสวัสดีค่...


## Inspect Results

The Markdown form is easier to scan and can be saved locally when `SAVE_OUTPUTS = True`.

In [4]:
transcript_markdown = "\n".join(
    f"- [{row.timestamp}] {row.text}" for row in transcript_df.itertuples(index=False)
)

print(transcript_markdown)

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "transcript.json").write_text(
        json.dumps(rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (OUTPUT_DIR / "transcript.md").write_text(transcript_markdown, encoding="utf-8")
    print("Saved transcript files to:", OUTPUT_DIR)

- [00:00:00 - 00:00:06.830] สวัสดีครับ ผมม38ครับสวัสดีครับ ม39ครับสวัสดีค่ะ F42ค่ะ
